# 12 — Decomposição Temporal: Tempo de Resolução por Assunto

**Objetivo:** Decompor o tempo de resolução por assunto do ticket para inferir complexidade e penalidade de transferência entre departamentos. Classificar assuntos por padrão de duração e quantificar o custo operacional do roteamento ineficiente.

**Dados:** `customer_support_tickets.csv` — 8.469 tickets, 2.769 fechados com CSAT.  
**Métrica-chave:** `duration_hours = abs(TTR - FRT)` em horas (proxy para esforço pós-contato).

**Premissa:** Assuntos com durações sistematicamente mais altas provavelmente envolvem transferência entre departamentos ou escalonamento — o tempo extra é uma "penalidade de roteamento" que pode ser reduzida com auto-roteamento inteligente.

---

## Seção 1: Análise por Assunto do Ticket

Carregamos os dados, calculamos `duration_hours`, e analisamos como os 16 assuntos se distribuem entre os 4 tipos de ticket. Em seguida, para cada assunto: contagem, mediana de FRT, mediana de duração, CSAT médio e desvio-padrão de duração.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
csv_path = 'data/customer_support_tickets.csv'
df_raw = pd.read_csv(csv_path)
print(f'Dataset total: {len(df_raw)} tickets')

# Filter closed tickets with CSAT
df = df_raw[df_raw['Ticket Status'] == 'Closed'].copy()
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'], errors='coerce')
df['duration_hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds().abs() / 3600
df['frt_hours'] = (df['First Response Time'] - df['First Response Time'].min()).dt.total_seconds() / 3600  # relative FRT
df = df.dropna(subset=['Customer Satisfaction Rating', 'duration_hours'])

df = df.rename(columns={
    'Ticket Channel': 'channel',
    'Ticket Subject': 'subject',
    'Ticket Type': 'type',
    'Customer Satisfaction Rating': 'csat',
    'Ticket Priority': 'priority',
})

print(f'Tickets fechados com CSAT e duração: {len(df)}')
print(f'Horas totais operacionais: {df["duration_hours"].sum():,.0f}h')
print(f'Assuntos: {len(df["subject"].unique())} | Tipos: {len(df["type"].unique())} | Canais: {len(df["channel"].unique())}')

In [ ]:
# Cross-tabulation: Subject × Type
ct = pd.crosstab(df['subject'], df['type'], margins=True, margins_name='Total')
print('Tabulação cruzada: Assunto × Tipo de Ticket')
print('=' * 80)
print(ct.to_string())
print()

# Check if subjects are tied to specific types
print('\nAnálise: Os assuntos estão vinculados a tipos específicos?')
for subj in sorted(df['subject'].unique()):
    subj_data = df[df['subject'] == subj]
    type_dist = subj_data['type'].value_counts(normalize=True)
    dominant = type_dist.index[0]
    dominant_pct = type_dist.values[0] * 100
    n_types = len(type_dist)
    marker = '⚠' if dominant_pct > 60 else '✓'
    print(f'  {marker} {subj}: {n_types} tipos — {dominant} = {dominant_pct:.0f}%')

In [ ]:
# Per-subject statistics table
subject_stats = df.groupby('subject').agg(
    n_tickets=('duration_hours', 'count'),
    median_frt_h=('frt_hours', 'median'),
    median_duration_h=('duration_hours', 'median'),
    mean_duration_h=('duration_hours', 'mean'),
    std_duration_h=('duration_hours', 'std'),
    mean_csat=('csat', 'mean'),
).round(2)

subject_stats = subject_stats.sort_values('median_duration_h', ascending=False)

print('Estatísticas por Assunto (ordenado por mediana de duração decrescente)')
print('=' * 120)
print(f'{"Assunto":<30} {"N":>5} {"Med FRT(h)":>10} {"Med Dur(h)":>10} {"Média Dur":>10} {"Std Dur":>8} {"CSAT Méd":>9}')
print('-' * 120)
for subj, row in subject_stats.iterrows():
    print(f'{subj:<30} {row.n_tickets:>5.0f} {row.median_frt_h:>10.1f} {row.median_duration_h:>10.1f} {row.mean_duration_h:>10.1f} {row.std_duration_h:>8.1f} {row.mean_csat:>9.2f}')
print('-' * 120)

# Global stats
global_median = df['duration_hours'].median()
global_mean = df['duration_hours'].mean()
print(f'\nMediana global de duração: {global_median:.1f}h')
print(f'Média global de duração: {global_mean:.1f}h')
print(f'CSAT médio global: {df["csat"].mean():.2f}')

## Seção 2: Classificação por Complexidade

Usamos a distribuição de medianas de duração entre os 16 assuntos para classificar cada um:

- **Handler único provável** — mediana de duração abaixo da mediana de todas as medianas. Indica resolução simples, sem necessidade de escalonamento.
- **Transferência provável** — mediana de duração acima do P75 de todas as medianas. Indica que o ticket provavelmente passa por mais de um agente ou departamento.
- **Ambíguo** — entre a mediana e o P75. Pode ser complexidade intrínseca do assunto ou transferência ocasional.

**Raciocínio:** Se um assunto tem mediana de resolução sistematicamente mais alta que outros, isso sugere que o processo de resolução envolve mais etapas — não que os tickets individuais sejam mais difíceis. Em dados reais com timestamps de transferência, poderíamos verificar diretamente. Com dados sintéticos, a variância entre assuntos é o melhor proxy disponível.

In [ ]:
# Classify subjects by complexity
medians = subject_stats['median_duration_h']
median_of_medians = medians.median()
p75_of_medians = medians.quantile(0.75)

print(f'Mediana das medianas de duração: {median_of_medians:.1f}h')
print(f'P75 das medianas de duração: {p75_of_medians:.1f}h')
print()

def classify_complexity(med_dur):
    if med_dur <= median_of_medians:
        return 'Handler único provável'
    elif med_dur >= p75_of_medians:
        return 'Transferência provável'
    else:
        return 'Ambíguo'

subject_stats['complexity'] = subject_stats['median_duration_h'].apply(classify_complexity)

# Color-coded display
COMPLEXITY_COLORS = {
    'Handler único provável': '\033[92m',  # green
    'Ambíguo': '\033[93m',                  # yellow
    'Transferência provável': '\033[91m',   # red
}
RESET = '\033[0m'

print('Classificação de Complexidade por Assunto')
print('=' * 100)
print(f'{"Assunto":<30} {"Med Dur(h)":>10} {"Classificação":<30}')
print('-' * 100)
for subj, row in subject_stats.iterrows():
    color = COMPLEXITY_COLORS.get(row.complexity, '')
    print(f'{subj:<30} {row.median_duration_h:>10.1f} {color}{row.complexity:<30}{RESET}')
print('-' * 100)

# Summary counts
print('\nResumo:')
for cat in ['Handler único provável', 'Ambíguo', 'Transferência provável']:
    subjects_in_cat = subject_stats[subject_stats['complexity'] == cat]
    n = len(subjects_in_cat)
    total_tickets = subjects_in_cat['n_tickets'].sum()
    print(f'  {cat}: {n} assuntos, {total_tickets:.0f} tickets')

# Store for later use
single_handler_subjects = subject_stats[subject_stats['complexity'] == 'Handler único provável'].index.tolist()
transfer_subjects = subject_stats[subject_stats['complexity'] == 'Transferência provável'].index.tolist()
ambiguous_subjects = subject_stats[subject_stats['complexity'] == 'Ambíguo'].index.tolist()

print(f'\nHandler único: {single_handler_subjects}')
print(f'Transferência: {transfer_subjects}')
print(f'Ambíguo: {ambiguous_subjects}')

## Seção 3: Decomposição Temporal por Assunto

Modelo de decomposição:
- **Baseline de atendimento** = mediana de duração dos assuntos "handler único" (resolução simples, sem transferência)
- **Penalidade de transferência** = mediana do assunto − baseline (tempo extra atribuído a roteamento/escalonamento)

O gráfico de barras empilhadas mostra visualmente quanto do tempo de cada assunto é "tempo produtivo de resolução" (azul) vs "penalidade de roteamento" (vermelho).

In [ ]:
# Baseline = median duration of single-handler subjects
baseline_durations = subject_stats.loc[single_handler_subjects, 'median_duration_h']
baseline = baseline_durations.median()
print(f'Baseline de atendimento (mediana dos "handler único"): {baseline:.1f}h')
print()

# Compute transfer penalty for each subject
subject_stats['baseline_handling'] = baseline
subject_stats['transfer_penalty'] = (subject_stats['median_duration_h'] - baseline).clip(lower=0)

# Sort by total (median) duration for the chart
chart_data = subject_stats.sort_values('median_duration_h', ascending=True).copy()

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 10))

y_pos = np.arange(len(chart_data))
bar_height = 0.7

# Baseline bars (blue)
bars_base = ax.barh(y_pos, chart_data['baseline_handling'], height=bar_height,
                     color='#3b82f6', label=f'Tempo base de resolução ({baseline:.0f}h)', edgecolor='white')

# Transfer penalty bars (red, stacked)
bars_penalty = ax.barh(y_pos, chart_data['transfer_penalty'], height=bar_height,
                        left=chart_data['baseline_handling'],
                        color='#ef4444', label='Penalidade de transferência', edgecolor='white')

# Labels with complexity classification
for i, (subj, row) in enumerate(chart_data.iterrows()):
    total = row['median_duration_h']
    penalty = row['transfer_penalty']
    complexity_label = row['complexity'].split(' ')[0]  # First word
    ax.text(total + 5, i, f'{total:.0f}h (+{penalty:.0f}h) — {row.complexity}',
            va='center', fontsize=8.5,
            color='#ef4444' if row.complexity == 'Transferência provável'
                  else '#f59e0b' if row.complexity == 'Ambíguo'
                  else '#22c55e')

ax.set_yticks(y_pos)
ax.set_yticklabels(chart_data.index, fontsize=9)
ax.set_xlabel('Mediana de Duração (horas)', fontsize=11)
ax.set_title('Decomposição Temporal: Tempo de Resolução vs Penalidade de Transferência', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.axvline(x=baseline, color='#3b82f6', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.savefig('process-log/screenshots/p13_time_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nGráfico salvo em process-log/screenshots/p13_time_decomposition.png')
print(f'\nPenalidades de transferência:')
for subj, row in subject_stats.sort_values('transfer_penalty', ascending=False).iterrows():
    if row['transfer_penalty'] > 0:
        print(f'  {subj}: +{row["transfer_penalty"]:.1f}h de penalidade ({row["complexity"]})')

## Seção 4: Impacto por Canal × Assunto

Para os assuntos classificados como "transferência provável", analisamos como a duração varia por canal. Se o canal influencia a duração para assuntos complexos, isso confirma que o roteamento inicial afeta o tempo total — e que auto-roteamento inteligente pode reduzir a penalidade.

In [ ]:
# Heatmap: Subject (rows) × Channel (columns), values = median duration_hours
# Include transfer + ambiguous subjects for context
subjects_of_interest = transfer_subjects + ambiguous_subjects + single_handler_subjects

# Build pivot with ALL subjects for the heatmap
pivot_duration = df.groupby(['subject', 'channel'])['duration_hours'].median().unstack()
pivot_duration = pivot_duration.reindex(subject_stats.sort_values('median_duration_h', ascending=False).index)

fig, ax = plt.subplots(figsize=(12, 10))

# Annotate with formatted hours
sns.heatmap(pivot_duration, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mediana de Duração (horas)'},
            vmin=pivot_duration.min().min(), vmax=pivot_duration.max().max())

# Mark transfer subjects with red labels
ytick_labels = ax.get_yticklabels()
for label in ytick_labels:
    subj = label.get_text()
    if subj in transfer_subjects:
        label.set_color('#ef4444')
        label.set_fontweight('bold')
    elif subj in ambiguous_subjects:
        label.set_color('#f59e0b')

ax.set_title('Mediana de Duração (horas) por Assunto × Canal\n(vermelho = transferência provável, amarelo = ambíguo)', fontsize=12)
ax.set_xlabel('Canal', fontsize=11)
ax.set_ylabel('Assunto', fontsize=11)

plt.tight_layout()
plt.savefig('process-log/screenshots/p13_duration_heatmap_subject_channel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Heatmap salvo em process-log/screenshots/p13_duration_heatmap_subject_channel.png')

# Print variance analysis for transfer subjects
print('\nVariância de duração por canal (assuntos de transferência provável):')
print(f'{"Assunto":<30} {"Min Canal":>10} {"Max Canal":>10} {"Amplitude":>10} {"CV":>8}')
print('-' * 80)
for subj in transfer_subjects:
    subj_data = pivot_duration.loc[subj].dropna()
    if len(subj_data) > 1:
        min_ch = subj_data.idxmin()
        max_ch = subj_data.idxmax()
        amplitude = subj_data.max() - subj_data.min()
        cv = subj_data.std() / subj_data.mean() * 100 if subj_data.mean() > 0 else 0
        print(f'{subj:<30} {subj_data.min():>10.0f} {subj_data.max():>10.0f} {amplitude:>10.0f} {cv:>7.1f}%')
        print(f'  {"":>30} ({min_ch}) ({max_ch})')

## Seção 5: Quantificação da Penalidade de Roteamento

Quanto tempo total é perdido com a penalidade de transferência? Para cada ticket fechado, calculamos:
- Se o assunto tem penalidade > 0: `penalidade_ticket = penalidade_mediana_do_assunto`
- Total de horas de penalidade = soma de todas as penalidades individuais

Isso quantifica o **business case para auto-roteamento**: se eliminássemos a penalidade de transferência com roteamento inteligente, quanto tempo economizaríamos?

In [ ]:
# Map transfer penalty to each ticket
penalty_map = subject_stats['transfer_penalty'].to_dict()
df['transfer_penalty_h'] = df['subject'].map(penalty_map)

total_operational_hours = 21439  # from notebook 08
total_closed_hours = df['duration_hours'].sum()

# Total penalty hours
total_penalty_hours = df['transfer_penalty_h'].sum()
penalty_pct_of_total = total_penalty_hours / total_operational_hours * 100
penalty_pct_of_closed = total_penalty_hours / total_closed_hours * 100

print('=' * 80)
print('QUANTIFICAÇÃO DA PENALIDADE DE ROTEAMENTO')
print('=' * 80)
print(f'Total de horas operacionais (todos os tickets fechados): {total_operational_hours:,.0f}h')
print(f'Total de horas dos tickets com CSAT: {total_closed_hours:,.0f}h')
print(f'Total de horas em penalidade de transferência: {total_penalty_hours:,.0f}h')
print(f'  → {penalty_pct_of_total:.1f}% do total operacional ({total_operational_hours:,}h)')
print(f'  → {penalty_pct_of_closed:.1f}% dos tickets fechados com CSAT ({total_closed_hours:,.0f}h)')
print()

# Per-subject breakdown
print('Breakdown por Assunto:')
print(f'{"Assunto":<30} {"Tickets":>8} {"Penalidade/Ticket":>18} {"Total Penalidade":>16} {"% do Total":>10}')
print('-' * 90)
for subj in subject_stats.sort_values('transfer_penalty', ascending=False).index:
    pen = subject_stats.loc[subj, 'transfer_penalty']
    if pen > 0:
        n = int(subject_stats.loc[subj, 'n_tickets'])
        total_pen = n * pen
        pct = total_pen / total_penalty_hours * 100 if total_penalty_hours > 0 else 0
        print(f'{subj:<30} {n:>8} {pen:>18.1f}h {total_pen:>16,.0f}h {pct:>9.1f}%')
print('-' * 90)

# Visualization: waterfall-style showing penalty contribution
penalty_by_subject = []
for subj in subject_stats.sort_values('transfer_penalty', ascending=False).index:
    pen = subject_stats.loc[subj, 'transfer_penalty']
    if pen > 0:
        n = int(subject_stats.loc[subj, 'n_tickets'])
        penalty_by_subject.append({'subject': subj, 'penalty_hours': n * pen, 'n_tickets': n, 'penalty_per_ticket': pen})

penalty_df = pd.DataFrame(penalty_by_subject)

if len(penalty_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    penalty_df_sorted = penalty_df.sort_values('penalty_hours', ascending=True)
    bars = ax.barh(penalty_df_sorted['subject'], penalty_df_sorted['penalty_hours'],
                   color='#ef4444', edgecolor='white', alpha=0.85)
    for bar, (_, row) in zip(bars, penalty_df_sorted.iterrows()):
        ax.text(bar.get_width() + total_penalty_hours * 0.01, bar.get_y() + bar.get_height()/2,
                f'{row.penalty_hours:,.0f}h ({row.n_tickets} tickets × {row.penalty_per_ticket:.0f}h)',
                va='center', fontsize=9)
    ax.set_xlabel('Total de Horas de Penalidade', fontsize=11)
    ax.set_title(f'Penalidade de Roteamento por Assunto\nTotal: {total_penalty_hours:,.0f}h ({penalty_pct_of_total:.1f}% de {total_operational_hours:,}h operacionais)',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0, penalty_df['penalty_hours'].max() * 1.4)
    plt.tight_layout()
    plt.savefig('process-log/screenshots/p13_routing_penalty_by_subject.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nGráfico salvo em process-log/screenshots/p13_routing_penalty_by_subject.png')

In [ ]:
# Business case summary
print()
print('=' * 80)
print('BUSINESS CASE PARA AUTO-ROTEAMENTO')
print('=' * 80)
print(f'''
Este é o business case para auto-roteamento:

  {total_penalty_hours:,.0f} horas são perdidas esperando transferência entre departamentos.
  
  Isso representa {penalty_pct_of_total:.1f}% das {total_operational_hours:,} horas operacionais totais.
  
  Com auto-roteamento inteligente (classificação automática do assunto + 
  direcionamento ao especialista correto desde o primeiro contato), é possível
  eliminar parcial ou totalmente essa penalidade.
  
  Cenários de redução:
  - Conservador (30% redução): -{total_penalty_hours * 0.30:,.0f}h/período
  - Moderado (50% redução):    -{total_penalty_hours * 0.50:,.0f}h/período
  - Agressivo (70% redução):   -{total_penalty_hours * 0.70:,.0f}h/período
  
  NOTA: Os dados são sintéticos — os valores absolutos são ilustrativos.
  A METODOLOGIA (decomposição temporal + classificação de complexidade) é o 
  entregável real. Com dados reais, os mesmos cálculos produziriam estimativas
  acionáveis.
''')

## Seção 6: Conclusões

### Por que decompor o tempo desta forma?

A métrica `duration_hours = abs(TTR - FRT)` é um proxy imperfeito mas útil. Ela captura o tempo total entre a primeira resposta do agente e a resolução final. Em um fluxo real de suporte:

- **Tickets simples:** O mesmo agente que responde primeiro resolve o ticket. O tempo é predominantemente de trabalho ativo.
- **Tickets complexos:** O ticket é transferido para outro departamento, escalado a um supervisor, ou passa por múltiplas interações. O tempo inclui espera em filas, handoffs, e re-leitura do contexto pelo novo agente.

A diferença de mediana entre assuntos captura essa dinâmica: assuntos que sistematicamente demoram mais provavelmente envolvem mais transferências.

### Premissas e limitações

1. **Dados sintéticos:** Os timestamps FRT e TTR são randomizados, então as durações absolutas não refletem um processo real. A variância entre assuntos é o que importa, não os valores absolutos.
2. **Proxy de transferência:** Sem dados de handoff explícitos (ex: `transferred_at`, `escalated_to`), usamos a duração como proxy. Em dados reais, seria possível medir transferências diretamente.
3. **Baseline conservador:** Usamos a mediana dos assuntos "handler único" como baseline. Isso assume que o tempo mínimo de resolução é representado por esses assuntos.
4. **Uniformidade:** Os dados sintéticos têm distribuições relativamente uniformes (~25% por canal, ~6% por assunto), o que limita a diferenciação. Com dados reais, as diferenças seriam provavelmente mais pronunciadas.

### Conexão com o framework de 6 cenários

A decomposição temporal complementa o diagnóstico de 6 cenários (notebook 08):

| Cenário | Relação com decomposição temporal |
|---------|----------------------------------|
| **Acelerar** | Pares com penalidade alta E correlação negativa (r < -0.3) → auto-roteamento + SLA agressivo |
| **Desacelerar** | Pares com penalidade alta E correlação positiva (r > 0.3) → routing para especialista, mais tempo = melhor |
| **Redirecionar** | Canal com penalidade desproporcional → mover para canal com menor penalidade |
| **Quarentena** | Alta penalidade, baixo CSAT, sem canal viável → investigar causa raiz |
| **Manter** | Baixa penalidade, CSAT bom → não mexer |
| **Liberar** | Baixa penalidade, sem correlação → reduzir esforço, automatizar |

### Achado principal

**A penalidade de transferência quantifica a oportunidade de automação.** Cada hora de penalidade é uma hora que poderia ser eliminada com roteamento inteligente — sem mudar a qualidade do atendimento, apenas eliminando o tempo de espera entre departamentos.

In [ ]:
# Final summary print
print('=' * 80)
print('RESUMO — DECOMPOSIÇÃO TEMPORAL')
print('=' * 80)
print(f'''
Dados analisados: {len(df)} tickets fechados com CSAT
Horas operacionais totais (referência): {total_operational_hours:,}h
Baseline de resolução (handler único): {baseline:.1f}h

Classificação de complexidade:
  - Handler único provável: {len(single_handler_subjects)} assuntos
  - Ambíguo: {len(ambiguous_subjects)} assuntos
  - Transferência provável: {len(transfer_subjects)} assuntos

Penalidade de roteamento:
  - Total: {total_penalty_hours:,.0f}h ({penalty_pct_of_total:.1f}% do operacional)
  - Assuntos com penalidade: {len(penalty_df) if len(penalty_df) > 0 else 0}

Implicação para auto-roteamento:
  - {total_penalty_hours:,.0f}h podem ser parcialmente eliminadas
  - Com 50% de redução: {total_penalty_hours * 0.50:,.0f}h economizadas
  - Metodologia validada — aplicável a dados reais
''')
print('=' * 80)
print('Notebook 12 concluído com sucesso.')